# Orion Dense vs Window vs Sparse Experiment Notebook (Colab Ready)

This notebook is a robust **research-grade comparison harness** for dense, window, and sparse attention in Orion.

It includes fixes for common experimental pitfalls:

- token-matched budgets (not step-matched)
- multi-seed statistics
- consistent TinyShakespeare validation metric
- OOM/CUDA-failure resilient trial execution (auto batch-size backoff)
- model-only throughput microbench (to reduce data-pipeline confounding)
- hardware/runtime metadata logging
- paired non-dense-vs-dense analysis (no "best sparse vs single dense" selection bias)
- flex sparse probe metrics support (`sparse_probe_every`, `sparse_probe_tokens`)


## 1) Environment Setup


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/akashkguw/orion.git"
REPO_ROOT = Path("/content/orion") if IN_COLAB else Path.cwd()

if IN_COLAB and not REPO_ROOT.exists():
    print(f"Cloning {REPO_URL} -> {REPO_ROOT}")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)

os.chdir(REPO_ROOT)
print(f"IN_COLAB={IN_COLAB}")
print(f"REPO_ROOT={REPO_ROOT}")

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        "requirements.txt",
        "-r",
        "requirements-dev.txt",
    ],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "seaborn"], check=True)

print("Dependencies installed.")

In [ ]:
import itertools
import json
import math
import platform
import subprocess
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)

## 2) Hardware / Runtime Metadata


In [ ]:
def get_git_commit() -> str:
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    except Exception:
        return "unknown"


def get_hardware_meta() -> dict:
    gpu_name = "cpu"
    gpu_count = 0
    cuda_version = None
    if torch.cuda.is_available():
        gpu_count = torch.cuda.device_count()
        gpu_name = torch.cuda.get_device_name(0)
        cuda_version = torch.version.cuda

    return {
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "torch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": cuda_version,
        "gpu_count": gpu_count,
        "gpu_name": gpu_name,
        "git_commit": get_git_commit(),
    }


HARDWARE_META = get_hardware_meta()
HARDWARE_META

## 3) Configure Sweep

Use `PROFILE='pilot9'` for a fast, balanced sanity experiment with exactly **9 trials**:
- 3 dense baselines (one per sequence length)
- 3 window runs
- 3 sparse runs

Use `PROFILE='pilot'` for a broader sweep and `PROFILE='full'` for large-scale runs.

Important fairness design:
- **Token-matched** budgets across `seq_len`.
- Dense, window, and sparse all use the same `(seq_len, seed, lr)` anchor in each trial group.
- Analysis is paired on `(seq_len, seed, lr)` against dense baselines.


In [ ]:
PROFILE = "pilot9"  # "pilot9", "pilot", or "full"

if PROFILE == "pilot9":
    TRAIN_TOKENS_TARGET = 4_000_000
    SEEDS = [123]
    SEQ_LENS = [256, 512, 1024]
    LR_GRID = [3e-4]
    SPARSE_GRID = [
        (64, 8),
    ]
    WINDOW_GRID = [128]
    BATCH_CANDIDATES_BY_SEQ = {
        256: [16, 12, 8, 4],
        512: [8, 6, 4, 2],
        1024: [4, 3, 2, 1],
    }
elif PROFILE == "pilot":
    TRAIN_TOKENS_TARGET = 6_000_000
    SEEDS = [123, 456, 789]
    SEQ_LENS = [256, 512, 1024]
    LR_GRID = [3e-4, 2e-4]
    SPARSE_GRID = [
        (64, 8),
        (128, 16),
    ]
    WINDOW_GRID = [64, 128]  # window sizes to sweep
    BATCH_CANDIDATES_BY_SEQ = {
        256: [16, 12, 8, 4],
        512: [8, 6, 4, 2],
        1024: [4, 3, 2, 1],
    }
elif PROFILE == "full":
    TRAIN_TOKENS_TARGET = 12_000_000
    SEEDS = [123, 456, 789]
    SEQ_LENS = [256, 512, 1024, 2048, 4096]
    LR_GRID = [3e-4, 2e-4]
    SPARSE_GRID = [
        (64, 8),
        (128, 16),
        (256, 16),
    ]
    WINDOW_GRID = [64, 128, 256]
    BATCH_CANDIDATES_BY_SEQ = {
        256: [16, 12, 8, 4],
        512: [8, 6, 4, 2],
        1024: [4, 3, 2, 1],
        2048: [2, 1],
        4096: [1],
    }
else:
    raise ValueError(f"Unknown PROFILE: {PROFILE}")

BASE_MODEL = {
    "name": "orion",
    "d_model": 256,
    "n_layers": 4,
    "n_heads": 4,
    "mlp_mult": 4,
}

# Evaluation / benchmark controls
RUN_VAL_EVAL = True
VAL_EVAL_BATCHES = 40
RUN_MODEL_ONLY_BENCH = True
BENCH_WARMUP = 5
BENCH_ITERS = 20

# Runner controls
AUTO_RETRY_OOM = True
AUTO_RETRY_DENSE_CUDA_FAIL = True
OVERWRITE = False

# Optional sparse telemetry probes for flex-only runs.
SPARSE_PROBE_EVERY = 50
SPARSE_PROBE_TOKENS = 256

EXPERIMENT_ID = time.strftime("dense_sparse_research_%Y%m%d_%H%M%S")
RUNS_ROOT = Path("runs") / EXPERIMENT_ID
CFG_ROOT = Path("configs") / "generated" / EXPERIMENT_ID
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
CFG_ROOT.mkdir(parents=True, exist_ok=True)

with (RUNS_ROOT / "hardware_meta.json").open("w", encoding="utf-8") as f:
    json.dump(HARDWARE_META, f, indent=2)

print("PROFILE:", PROFILE)
print("TRAIN_TOKENS_TARGET:", TRAIN_TOKENS_TARGET)
print("RUNS_ROOT:", RUNS_ROOT)
print("CFG_ROOT:", CFG_ROOT)


In [ ]:
@dataclass(frozen=True)
class TrialSpec:
    backend: str  # "dense" | "window" | "sparse"
    seq_len: int
    seed: int
    lr: float
    sparse_tag: str = "dense"  # "dense", "w{W}" (window), or "w{W}_d{D}" (sparse)

    @property
    def trial_id(self) -> str:
        lr_tag = str(self.lr).replace(".", "p")
        return f"{self.backend}_T{self.seq_len}_seed{self.seed}_lr{lr_tag}_{self.sparse_tag}"


def build_trial_specs() -> list[TrialSpec]:
    specs: list[TrialSpec] = []

    for seed, seq_len, lr in itertools.product(SEEDS, SEQ_LENS, LR_GRID):
        # Dense baseline
        specs.append(
            TrialSpec(backend="dense", seq_len=seq_len, seed=seed, lr=lr, sparse_tag="dense")
        )

        # Window attention
        for window_size in WINDOW_GRID:
            if window_size >= seq_len:
                continue
            tag = f"w{window_size}"
            specs.append(
                TrialSpec(backend="window", seq_len=seq_len, seed=seed, lr=lr, sparse_tag=tag)
            )

        # Sparse attention
        for window_size, expander_degree in SPARSE_GRID:
            if window_size >= seq_len:
                continue
            tag = f"w{window_size}_d{expander_degree}"
            specs.append(
                TrialSpec(backend="sparse", seq_len=seq_len, seed=seed, lr=lr, sparse_tag=tag)
            )

    return specs


TRIAL_SPECS = build_trial_specs()
print(f"Planned trials: {len(TRIAL_SPECS)}")
pd.DataFrame(asdict(s) for s in TRIAL_SPECS).head(20)

## 4) Runner + Metrics Utilities


In [ ]:
from orion.attention.base import AttentionConfig
from orion.data.shakespeare import load_tiny_shakespeare
from orion.model import loss_fn
from orion.models_factory import build_model


def steps_for_token_budget(seq_len: int, batch_size: int, token_budget: int) -> int:
    return max(1, math.ceil(token_budget / (seq_len * batch_size)))


def parse_sparse_tag(tag: str) -> tuple[int, int] | None:
    if tag == "dense":
        return None
    # format: w{W}_d{D}
    left, right = tag.split("_")
    return int(left[1:]), int(right[1:])


def attention_degree_from_spec(spec: TrialSpec) -> int:
    if spec.backend == "dense":
        return spec.seq_len
    if spec.backend == "window":
        return int(spec.sparse_tag[1:])  # tag is "w{W}"
    wd = parse_sparse_tag(spec.sparse_tag)
    assert wd is not None
    w, d = wd
    return w + d


def trial_to_config(spec: TrialSpec, out_dir: Path, batch_size: int) -> dict:
    steps = steps_for_token_budget(spec.seq_len, batch_size, TRAIN_TOKENS_TARGET)

    cfg = {
        "run": {
            "out_dir": str(out_dir),
            "seed": int(spec.seed),
            "steps": int(steps),
            "log_every": 10,
            "save_every": int(steps),
        },
        "data": {
            "dataset": "tinyshakespeare",
            "root": "data",
            "seq_len": int(spec.seq_len),
            "batch_size": int(batch_size),
        },
        "model": {
            "name": BASE_MODEL["name"],
            "d_model": int(BASE_MODEL["d_model"]),
            "n_layers": int(BASE_MODEL["n_layers"]),
            "n_heads": int(BASE_MODEL["n_heads"]),
            "mlp_mult": int(BASE_MODEL["mlp_mult"]),
        },
        "attention": {
            "backend": spec.backend,
        },
        "optim": {
            "lr": float(spec.lr),
        },
    }

    if spec.backend == "window":
        cfg["attention"]["window_size"] = int(spec.sparse_tag[1:])  # tag is "w{W}"
    elif spec.backend == "sparse":
        w, d = parse_sparse_tag(spec.sparse_tag)
        cfg["attention"]["window_size"] = int(w)
        cfg["attention"]["expander_degree"] = int(d)
        cfg["attention"]["sparse_impl"] = "flex"
        cfg["attention"]["sparse_block_size"] = 128
        cfg["attention"]["sparse_probe_every"] = int(SPARSE_PROBE_EVERY)
        cfg["attention"]["sparse_probe_tokens"] = int(SPARSE_PROBE_TOKENS)

    return cfg


def load_metrics_rows(metrics_path: Path) -> list[dict]:
    rows = []
    if not metrics_path.exists():
        return rows
    with metrics_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def mean_key(rows: list[dict], key: str) -> float:
    vals = [float(r[key]) for r in rows if key in r]
    return float(np.mean(vals)) if vals else math.nan


def deterministic_sample_batch(
    ids: torch.Tensor, batch_size: int, seq_len: int, device: torch.device, *, gen: torch.Generator
):
    n = ids.numel()
    max_start = n - (seq_len + 1)
    starts = torch.randint(0, max_start, (batch_size,), generator=gen)
    x = torch.stack([ids[s : s + seq_len] for s in starts], dim=0).to(device)
    y = torch.stack([ids[s + 1 : s + 1 + seq_len] for s in starts], dim=0).to(device)
    return x, y


def load_model_from_cfg_and_ckpt(cfg: dict, ckpt_path: Path, device: torch.device):
    # TinyShakespeare vocab comes from dataset.
    _, _, tok = load_tiny_shakespeare(cfg["data"].get("root", "data"))
    vocab_size = tok.vocab_size

    att = cfg.get("attention", {})
    attention_cfg = AttentionConfig(
        backend=str(att.get("backend", "dense")),
        window_size=int(att["window_size"]) if "window_size" in att else None,
        expander_degree=int(att["expander_degree"]) if "expander_degree" in att else None,
        sparse_impl=str(att.get("sparse_impl", "flex")),
        sparse_block_size=int(att.get("sparse_block_size", 128)),
        sparse_probe_every=int(att.get("sparse_probe_every", 0)),
        sparse_probe_tokens=int(att.get("sparse_probe_tokens", 256)),
    )

    model = build_model(
        name=cfg["model"]["name"],
        vocab_size=vocab_size,
        d_model=int(cfg["model"]["d_model"]),
        n_layers=int(cfg["model"]["n_layers"]),
        n_heads=int(cfg["model"]["n_heads"]),
        mlp_mult=int(cfg["model"]["mlp_mult"]),
        device=device,
        attention_cfg=attention_cfg,
    )

    ckpt = torch.load(str(ckpt_path), map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"], strict=True)
    model.eval()
    return model, tok


def evaluate_val_ppl(cfg: dict, ckpt_path: Path, *, batches: int = 40) -> tuple[float, float]:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, tok = load_model_from_cfg_and_ckpt(cfg, ckpt_path, device)

    _, val_ids, _ = load_tiny_shakespeare(cfg["data"].get("root", "data"))
    batch_size = int(cfg["data"]["batch_size"])
    seq_len = int(cfg["data"]["seq_len"])

    gen = torch.Generator(device="cpu")
    gen.manual_seed(2026)

    losses = []
    with torch.no_grad():
        for _ in range(batches):
            x, y = deterministic_sample_batch(val_ids, batch_size, seq_len, device, gen=gen)
            logits = model(x)
            loss = loss_fn(logits, y)
            losses.append(float(loss.item()))

    val_loss = float(np.mean(losses))
    val_ppl = float(math.exp(min(val_loss, 100.0)))
    return val_loss, val_ppl


def benchmark_model_only_forward(
    cfg: dict, ckpt_path: Path, *, warmup: int = 5, iters: int = 20
) -> float:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, tok = load_model_from_cfg_and_ckpt(cfg, ckpt_path, device)

    batch_size = int(cfg["data"]["batch_size"])
    seq_len = int(cfg["data"]["seq_len"])
    vocab_size = tok.vocab_size

    x = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)

    for _ in range(warmup):
        _ = model(x)

    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(iters):
        _ = model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    dt = time.time() - t0

    tokens = iters * batch_size * seq_len
    return tokens / dt if dt > 0 else math.nan

In [ ]:
def summarize_run(
    spec: TrialSpec, cfg: dict, run_dir: Path, duration_s: float, status: str, error: str = ""
) -> dict:
    out = {
        "trial_id": spec.trial_id,
        "backend": spec.backend,
        "sparse_tag": spec.sparse_tag,
        "seq_len": spec.seq_len,
        "seed": spec.seed,
        "lr": spec.lr,
        "batch_size": int(cfg["data"]["batch_size"]),
        "steps_target": int(cfg["run"]["steps"]),
        "train_tokens_target": TRAIN_TOKENS_TARGET,
        "attention_degree_expected": attention_degree_from_spec(spec),
        "duration_s": duration_s,
        "status": status,
        "error": error,
        "run_dir": str(run_dir),
        **HARDWARE_META,
    }

    metrics_path = run_dir / "metrics.jsonl"
    rows = load_metrics_rows(metrics_path)

    if status != "ok" or not rows:
        return out

    step_rows = [r for r in rows if r.get("type") == "step"]
    window_rows = [r for r in rows if r.get("type") == "window"]

    if not step_rows:
        out["status"] = "no_step_rows"
        return out

    final_step = step_rows[-1]
    tail = step_rows[-min(20, len(step_rows)) :]

    out.update(
        {
            "steps_observed": int(final_step.get("step", len(step_rows))),
            "final_loss": float(final_step.get("loss", math.nan)),
            "final_ppl": float(final_step.get("ppl", math.nan)),
            "final_acc_top1": float(final_step.get("accuracy_top1", math.nan)),
            "throughput_tok_s_tail_mean": mean_key(tail, "throughput_tokens_per_sec"),
            "step_time_ms_tail_mean": mean_key(tail, "step_time_ms"),
            "grad_norm_tail_mean": mean_key(tail, "grad_norm"),
        }
    )

    if window_rows:
        w = window_rows[-1]
        out.update(
            {
                "vram_peak_mib": float(w.get("vram_peak_mib", math.nan)),
                "attn_entropy": float(w.get("attention_entropy", math.nan)),
                "attn_entropy_norm": float(w.get("attention_entropy_normalized", math.nan)),
                "valid_neighbors": float(w.get("valid_neighbor_fraction", math.nan)),
                "window_mass_pct": float(w.get("attention_mass_window_pct", math.nan)),
                "expander_mass_pct": float(w.get("attention_mass_expander_pct", math.nan)),
                "valid_neighbor_fraction_causal_cap": float(
                    w.get("valid_neighbor_fraction_causal_cap", math.nan)
                ),
                "valid_neighbor_fraction_vs_causal_cap": float(
                    w.get("valid_neighbor_fraction_vs_causal_cap", math.nan)
                ),
                "future_neighbor_slots": float(w.get("future_neighbor_slots", math.nan)),
                "duplicate_neighbor_slots": float(w.get("duplicate_neighbor_slots", math.nan)),
            }
        )

    ckpt_path = run_dir / "checkpoint.pt"
    if RUN_VAL_EVAL and ckpt_path.exists():
        try:
            val_loss, val_ppl = evaluate_val_ppl(cfg, ckpt_path, batches=VAL_EVAL_BATCHES)
            out["val_loss"] = val_loss
            out["val_ppl"] = val_ppl
        except Exception as exc:
            out["val_loss"] = math.nan
            out["val_ppl"] = math.nan
            out["val_eval_error"] = str(exc)

    if RUN_MODEL_ONLY_BENCH and ckpt_path.exists():
        try:
            out["model_only_forward_tok_s"] = benchmark_model_only_forward(
                cfg, ckpt_path, warmup=BENCH_WARMUP, iters=BENCH_ITERS
            )
        except Exception as exc:
            out["model_only_forward_tok_s"] = math.nan
            out["bench_error"] = str(exc)

    return out


def is_oom_error(text: str) -> bool:
    t = (text or "").lower()
    patterns = [
        "cuda out of memory",
        "cuda error: out of memory",
        "torch.cuda.outofmemoryerror",
        "cudnn_status_alloc_failed",
        "cublas_status_alloc_failed",
        "std::bad_alloc",
        "out of memory",
        "resource exhausted",
    ]
    return any(p in t for p in patterns)


def is_retryable_cuda_failure(text: str) -> bool:
    t = (text or "").lower()
    patterns = [
        "cuda error",
        "cudnn",
        "cublas",
        "illegal memory access",
        "device-side assert",
        "resource exhausted",
    ]
    return any(p in t for p in patterns)


def run_trial(spec: TrialSpec, overwrite: bool = False) -> dict:
    run_dir = RUNS_ROOT / spec.trial_id
    run_dir.mkdir(parents=True, exist_ok=True)

    if spec.seq_len not in BATCH_CANDIDATES_BY_SEQ:
        err = (
            f"Missing BATCH_CANDIDATES_BY_SEQ entry for seq_len={spec.seq_len}. "
            f"Known keys: {sorted(BATCH_CANDIDATES_BY_SEQ.keys())}"
        )
        print(f"[fail] {spec.trial_id}: {err}")
        cfg = trial_to_config(spec, run_dir, batch_size=1)
        return summarize_run(spec, cfg, run_dir, duration_s=0.0, status="failed", error=err)

    metrics_path = run_dir / "metrics.jsonl"
    if metrics_path.exists() and not overwrite:
        # Reconstruct cfg from saved config if present.
        cfg_path_existing = run_dir / "config.yaml"
        if cfg_path_existing.exists():
            cfg = yaml.safe_load(cfg_path_existing.read_text())
        else:
            batch0 = BATCH_CANDIDATES_BY_SEQ[spec.seq_len][0]
            cfg = trial_to_config(spec, run_dir, batch0)
        print(f"[skip] {spec.trial_id}")
        return summarize_run(spec, cfg, run_dir, duration_s=0.0, status="ok")

    batch_candidates = list(BATCH_CANDIDATES_BY_SEQ[spec.seq_len])
    if 1 not in batch_candidates:
        batch_candidates.append(1)

    last_error = ""

    for attempt_idx, batch_size in enumerate(batch_candidates, start=1):
        cfg = trial_to_config(spec, run_dir, batch_size)
        cfg_path = run_dir / "config.yaml"
        cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")

        cmd = [sys.executable, "-m", "orion.train", "--config", str(cfg_path)]
        child_env = os.environ.copy()
        child_env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

        t0 = time.time()
        proc = subprocess.run(
            cmd,
            cwd=Path.cwd(),
            text=True,
            capture_output=True,
            env=child_env,
        )
        dt = time.time() - t0

        (run_dir / f"attempt_{attempt_idx}.stdout.log").write_text(proc.stdout, encoding="utf-8")
        (run_dir / f"attempt_{attempt_idx}.stderr.log").write_text(proc.stderr, encoding="utf-8")

        if proc.returncode == 0:
            print(f"[ok] {spec.trial_id} (bs={batch_size}, {dt:.1f}s)")
            return summarize_run(spec, cfg, run_dir, duration_s=dt, status="ok")

        err_blob = ((proc.stderr or "") + "\n" + (proc.stdout or ""))[-12000:]
        last_error = err_blob
        oom = is_oom_error(err_blob)

        retryable = AUTO_RETRY_OOM and oom
        if (
            not retryable
            and AUTO_RETRY_DENSE_CUDA_FAIL
            and spec.backend == "dense"
            and is_retryable_cuda_failure(err_blob)
        ):
            retryable = True

        if retryable and attempt_idx < len(batch_candidates):
            reason = ""
            for ln in reversed(err_blob.splitlines()):
                if ln.strip():
                    reason = ln.strip()
                    break
            print(
                f"[retry-backoff] {spec.trial_id}: bs={batch_size} failed, trying smaller batch"
                + (f" | tail: {reason[:180]}" if reason else "")
            )
            continue

        reason = ""
        for ln in reversed(err_blob.splitlines()):
            if ln.strip():
                reason = ln.strip()
                break
        print(
            f"[fail] {spec.trial_id} (bs={batch_size})"
            + (f" | tail: {reason[:220]}" if reason else "")
        )
        return summarize_run(spec, cfg, run_dir, duration_s=dt, status="failed", error=err_blob)

    # If loop exits unexpectedly, return failure.
    batch0 = batch_candidates[0] if batch_candidates else 1
    cfg = trial_to_config(spec, run_dir, batch0)
    return summarize_run(spec, cfg, run_dir, duration_s=0.0, status="failed", error=last_error)

## 5) Execute Sweep


In [ ]:
results = []
total = len(TRIAL_SPECS)

for i, spec in enumerate(TRIAL_SPECS, start=1):
    print(f"\n[{i}/{total}] {spec.trial_id}")
    row = run_trial(spec, overwrite=OVERWRITE)
    results.append(row)

summary_df = pd.DataFrame(results)
summary_path = RUNS_ROOT / "summary.csv"
summary_df.to_csv(summary_path, index=False)

print(f"\nSaved: {summary_path}")
print(summary_df["status"].value_counts(dropna=False))
summary_df.head(10)

## 6) Health Checks (Before Comparison)


In [ ]:
df = pd.read_csv(RUNS_ROOT / "summary.csv")
df_ok = df[df["status"] == "ok"].copy()

print("Successful trials:", len(df_ok), "/", len(df))

# Sparse structural sanity
sparse_ok = df_ok[df_ok["backend"] == "sparse"].copy()
if len(sparse_ok) > 0:
    print("\nSparse structural diagnostics summary:")
    display(
        sparse_ok[
            [
                "trial_id",
                "seq_len",
                "sparse_tag",
                "valid_neighbor_fraction_vs_causal_cap",
                "future_neighbor_slots",
                "duplicate_neighbor_slots",
                "window_mass_pct",
                "expander_mass_pct",
            ]
        ]
        .sort_values(["seq_len", "sparse_tag", "trial_id"])
        .head(30)
    )

    bad = sparse_ok[
        (sparse_ok["valid_neighbor_fraction_vs_causal_cap"] < 0.95)
        | (sparse_ok["future_neighbor_slots"] > 0)
        | (sparse_ok["duplicate_neighbor_slots"] > 0)
    ]
    print("Potential sparse-graph issues:", len(bad))

## 7) Paired Analysis: Dense vs Window vs Sparse

Pairing key: `(seq_len, seed, lr)`.


In [ ]:
# Build paired table: every non-dense trial matched to its dense baseline (same seq_len/seed/lr)

dense = df_ok[df_ok["backend"] == "dense"].copy()
non_dense = df_ok[df_ok["backend"] != "dense"].copy()

pair_keys = ["seq_len", "seed", "lr"]

dense_small = dense[
    pair_keys
    + [
        "trial_id",
        "train_tok_per_s",
        "model_tok_per_s",
        "val_ppl",
        "peak_vram_gb",
    ]
].rename(
    columns={
        "trial_id": "dense_trial_id",
        "train_tok_per_s": "train_tok_per_s_dense",
        "model_tok_per_s": "model_tok_per_s_dense",
        "val_ppl": "val_ppl_dense",
        "peak_vram_gb": "peak_vram_gb_dense",
    }
)

paired = non_dense.merge(dense_small, on=pair_keys, how="inner")
paired["speedup_train_over_dense"] = paired["train_tok_per_s"] / paired["train_tok_per_s_dense"]
paired["speedup_model_over_dense"] = paired["model_tok_per_s"] / paired["model_tok_per_s_dense"]
paired["vram_ratio_over_dense"] = paired["peak_vram_gb"] / paired["peak_vram_gb_dense"]
paired["val_ppl_delta"] = paired["val_ppl"] - paired["val_ppl_dense"]

print(f"Paired rows: {len(paired)}")
paired[["trial_id", "backend", "seq_len", "lr", "speedup_train_over_dense", "val_ppl_delta"]].head(
    20
)

In [ ]:
agg = paired.groupby(["backend", "seq_len", "sparse_tag", "lr"], as_index=False).agg(
    runs=("trial_id", "count"),
    train_speedup_mean=("speedup_train_over_dense", "mean"),
    train_speedup_std=("speedup_train_over_dense", "std"),
    model_speedup_mean=("speedup_model_over_dense", "mean"),
    vram_ratio_mean=("vram_ratio_over_dense", "mean"),
    val_ppl_delta_mean=("val_ppl_delta", "mean"),
    val_ppl_delta_std=("val_ppl_delta", "std"),
)

print(agg.to_string(index=False))

In [ ]:
# Suggested practical winner filter (edit thresholds as needed)
VAL_PPL_TOL = 0.20  # allow val ppl up to +0.20 vs dense

winners = agg[
    (agg["model_speedup_mean"] > 1.0)
    & (agg["vram_ratio_mean"] < 1.0)
    & (agg["val_ppl_delta_mean"] <= VAL_PPL_TOL)
].copy()

print("Candidate wins over dense (model speed + VRAM + quality):")
print(
    winners[
        [
            "backend",
            "seq_len",
            "sparse_tag",
            "lr",
            "model_speedup_mean",
            "vram_ratio_mean",
            "val_ppl_delta_mean",
        ]
    ].to_string(index=False)
)

## 8) Visualizations


In [ ]:
plot_df = paired.copy()
plot_df["variant"] = (
    plot_df["backend"] + "_" + plot_df["sparse_tag"] + "_lr" + plot_df["lr"].astype(str)
)

# Train throughput ratio by seq_len
plt.figure(figsize=(10, 5))
sns.lineplot(
    data=plot_df,
    x="seq_len",
    y="speedup_train_over_dense",
    hue="variant",
    marker="o",
    estimator="mean",
    errorbar="sd",
)
plt.axhline(1.0, color="black", linestyle="--", linewidth=1)
plt.title("Train Throughput Ratio vs Dense")
plt.ylabel("ratio (>1 means faster than dense)")
plt.xlabel("seq_len")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

# Model-only throughput ratio by seq_len
plt.figure(figsize=(10, 5))
sns.lineplot(
    data=plot_df,
    x="seq_len",
    y="speedup_model_over_dense",
    hue="variant",
    marker="o",
    estimator="mean",
    errorbar="sd",
)
plt.axhline(1.0, color="black", linestyle="--", linewidth=1)
plt.title("Model-Only Throughput Ratio vs Dense")
plt.ylabel("ratio (>1 means faster than dense)")
plt.xlabel("seq_len")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# VRAM ratio and validation quality delta (both vs dense)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.lineplot(
    data=plot_df,
    x="seq_len",
    y="vram_ratio_over_dense",
    hue="variant",
    marker="o",
    estimator="mean",
    errorbar="sd",
    ax=axes[0],
)
axes[0].axhline(1.0, color="black", linestyle="--", linewidth=1)
axes[0].set_title("VRAM Ratio vs Dense")
axes[0].set_ylabel("ratio (<1 means less VRAM than dense)")
axes[0].set_xlabel("seq_len")

sns.lineplot(
    data=plot_df,
    x="seq_len",
    y="val_ppl_delta",
    hue="variant",
    marker="o",
    estimator="mean",
    errorbar="sd",
    ax=axes[1],
)
axes[1].axhline(0.0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Validation PPL Delta vs Dense")
axes[1].set_ylabel("delta (<0 means better than dense)")
axes[1].set_xlabel("seq_len")

handles, labels = axes[1].get_legend_handles_labels()
axes[0].get_legend().remove() if axes[0].get_legend() else None
axes[1].get_legend().remove() if axes[1].get_legend() else None
fig.legend(handles, labels, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 9) Auto-Plot All Numeric Metrics

This section discovers every numeric metric in `summary.csv` and plots all non-constant metrics.

- Plots are saved to `runs/<EXPERIMENT_ID>/plots_all_metrics/`
- By default, all discovered metrics are rendered inline and saved


In [ ]:
import re

if "df_ok" not in globals():
    df = pd.read_csv(RUNS_ROOT / "summary.csv")
    df_ok = df[df["status"] == "ok"].copy()

df_plot = df_ok.copy()


# Keep variant labels compact but informative.
def _variant_label(row):
    lr = row.get("lr", np.nan)
    backend = row.get("backend", "unknown")
    if backend == "dense":
        return f"dense_lr{lr}"
    return f"{backend}_{row.get('sparse_tag', 'na')}_lr{lr}"


df_plot["variant"] = df_plot.apply(_variant_label, axis=1)

# Columns that are controls/ids rather than performance metrics.
control_numeric_cols = {
    "seq_len",
    "seed",
    "lr",
    "batch_size",
    "steps_target",
    "steps_observed",
    "train_tokens_target",
    "attention_degree_expected",
    "gpu_count",
    "cuda_available",
}

numeric_cols = [c for c in df_plot.columns if pd.api.types.is_numeric_dtype(df_plot[c])]

metric_cols = []
for c in numeric_cols:
    if c in control_numeric_cols:
        continue
    s = df_plot[c]
    if s.notna().sum() == 0:
        continue
    if s.nunique(dropna=True) <= 1:
        continue
    metric_cols.append(c)

metric_cols = sorted(metric_cols)

plots_dir = RUNS_ROOT / "plots_all_metrics"
plots_dir.mkdir(parents=True, exist_ok=True)

SHOW_INLINE = True
MAX_INLINE = None  # set e.g. 20 if you want fewer inline plots

print(f"Discovered numeric metrics: {len(metric_cols)}")
display(pd.DataFrame({"metric": metric_cols}))

rendered = 0
for idx, metric in enumerate(metric_cols, start=1):
    sub = df_plot[["seq_len", "variant", metric]].dropna()
    if sub.empty:
        continue

    plt.figure(figsize=(10, 5))
    if sub["seq_len"].nunique() > 1:
        sns.lineplot(
            data=sub,
            x="seq_len",
            y=metric,
            hue="variant",
            marker="o",
            estimator="mean",
            errorbar="sd",
        )
        plt.xlabel("seq_len")
    else:
        sns.boxplot(data=sub, x="variant", y=metric)
        plt.xticks(rotation=45, ha="right")
        plt.xlabel("variant")

    plt.title(f"{metric} (all variants)")
    plt.ylabel(metric)
    plt.tight_layout()

    safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", metric)
    out_file = plots_dir / f"{idx:03d}_{safe_name}.png"
    plt.savefig(out_file, dpi=170)

    rendered += 1
    if SHOW_INLINE and (MAX_INLINE is None or rendered <= MAX_INLINE):
        plt.show()
    else:
        plt.close()

print(f"Rendered and saved {rendered} metric plots to: {plots_dir}")

## 10) Notes / Caveats

- Dense may still dominate at short context due to highly optimized dense kernels.
- Window attention is usually a strong middle point: simpler than sparse, cheaper than dense.
- Sparse wins usually appear at larger `seq_len` with bounded sparse degree.
- Use `full` profile for stronger crossover evidence.
- Keep an eye on sparse diagnostics (`valid_neighbor_fraction_vs_causal_cap`, `future/duplicate edges`) before trusting speed/quality conclusions.
- In fused sparse mode, entropy/mass metrics can be `NA` unless probe metrics are enabled.
- Dense failures in Colab can be non-OOM CUDA errors; this notebook backs off batch size for retryable CUDA failures too.
